# Calendar / Dates Data Validation — PJM Like-Day Pipeline

**Source:** `{schema}.utils_v1_pjm_dates_daily` via `dates.pull_daily()`  
**Columns:** `date, day_of_week_number, is_weekend, is_nerc_holiday, summer_winter`  
**Feature builder:** `src.like_day_forecast.features.calendar_features` — 17 features  
**Purpose:** Validate raw calendar data and derived features before the like-day similarity pipeline.

### Checks
1. **Setup & Data Pull** — Pull dates, inspect shape and dtypes
2. **Completeness Checks** — Date gaps, duplicates, coverage of config start dates
3. **Column Value Checks** — DOW 0-6, is_weekend matches DOW 5/6, NERC holiday counts, summer/winter boundaries
4. **NERC Holiday Analysis** — List holidays, verify expected dates, weekend holidays
5. **Summer/Winter Transition** — Plot flag over time, verify PJM seasonal boundaries
6. **Feature Sanity Check** — Run `calendar_features.build()`, verify shapes and encodings
7. **Cyclical Encoding Visualization** — DOW and day-of-year circles
8. **DOW Group Distribution** — Bar chart of group sizes
9. **Recent Spot Check** — Last 14 days table
10. **Validation Summary** — Pass/fail checklist

## 1. Setup & Data Pull

In [ ]:
import sys
from pathlib import Path
_BACKEND = str(Path().resolve().parent.parent.parent.parent)
if _BACKEND not in sys.path:
    sys.path.insert(0, _BACKEND)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import src.like_day_forecast.settings
from src.like_day_forecast.data import dates
from src.like_day_forecast import configs

In [ ]:
df_dates = dates.pull_daily()

print(f"Shape: {df_dates.shape}")
print(f"Date range: {df_dates['date'].min()} to {df_dates['date'].max()}")
print(f"\nColumns: {list(df_dates.columns)}")
print(f"\nDtypes:\n{df_dates.dtypes}")
print(f"\nHead:")
df_dates.head(10)

### Previous 3 Days — All Calendar Components

Quick visual inspection of the most recent 3 days across all calendar fields: day-of-week, weekend flag, NERC holiday flag, and summer/winter designation.

In [ ]:
# Previous 3 days — all calendar components
df_last3 = df_dates.sort_values("date").tail(3).copy()
df_last3["_date_ts"] = pd.to_datetime(df_last3["date"])
df_last3["day_name"] = df_last3["_date_ts"].dt.day_name()

day_names_map = {0: "Mon", 1: "Tue", 2: "Wed", 3: "Thu", 4: "Fri", 5: "Sat", 6: "Sun"}

print("=== Day of Week — Previous 3 Days ===")
display(df_last3[["date", "day_name", "day_of_week_number"]])

print("\n=== Weekend Flag — Previous 3 Days ===")
display(df_last3[["date", "day_name", "is_weekend"]])

print("\n=== NERC Holiday Flag — Previous 3 Days ===")
display(df_last3[["date", "day_name", "is_nerc_holiday"]])

print("\n=== Summer/Winter Designation — Previous 3 Days ===")
display(df_last3[["date", "day_name", "summer_winter"]])

## 2. Completeness Checks

In [ ]:
# Check for duplicate dates
n_dupes = df_dates.duplicated(subset=["date"]).sum()
print(f"Duplicate dates: {n_dupes}")

# Check one row per day
date_series = pd.to_datetime(df_dates["date"])
full_range = pd.date_range(start=date_series.min(), end=date_series.max(), freq="D")
missing_dates = full_range.difference(date_series)
print(f"\nExpected days in range: {len(full_range)}")
print(f"Actual rows: {len(df_dates)}")
print(f"Missing dates (gaps): {len(missing_dates)}")
if len(missing_dates) > 0:
    print(f"  First 10 missing: {missing_dates[:10].tolist()}")

# Check coverage of config start dates
extended_start = pd.Timestamp(configs.EXTENDED_FEATURE_START)
full_start = pd.Timestamp(configs.FULL_FEATURE_START)
data_start = date_series.min()

print(f"\nConfig EXTENDED_FEATURE_START: {configs.EXTENDED_FEATURE_START}")
print(f"Config FULL_FEATURE_START:     {configs.FULL_FEATURE_START}")
print(f"Data starts at:                {data_start.date()}")
print(f"Covers EXTENDED_FEATURE_START: {'YES' if data_start <= extended_start else 'NO'}")
print(f"Covers FULL_FEATURE_START:     {'YES' if data_start <= full_start else 'NO'}")

## 3. Column Value Checks

In [ ]:
# day_of_week_number: verify values 0-6
dow_vals = sorted(df_dates["day_of_week_number"].unique())
print(f"day_of_week_number unique values: {dow_vals}")
print(f"Expected [0,1,2,3,4,5,6]: {'PASS' if dow_vals == list(range(7)) else 'FAIL'}")

# Distribution — verify DOW matches expected pattern (Mon=0 .. Sun=6)
dow_counts = df_dates["day_of_week_number"].value_counts().sort_index()
day_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
print(f"\nDOW distribution:")
for i, name in enumerate(day_names):
    print(f"  {i} ({name}): {dow_counts.get(i, 0):,}")

# Verify DOW assignment: check a known date
sample_date = pd.Timestamp("2024-01-01")  # Monday
sample_row = df_dates[pd.to_datetime(df_dates["date"]) == sample_date]
if len(sample_row) > 0:
    actual_dow = sample_row["day_of_week_number"].iloc[0]
    print(f"\nSpot check: 2024-01-01 (Monday) -> DOW={actual_dow} ({'PASS' if actual_dow == 0 else 'FAIL'})")

In [ ]:
# is_weekend: verify matches DOW 5,6
df_check = df_dates.copy()
df_check["_expected_weekend"] = df_check["day_of_week_number"].isin([5, 6])
df_check["_actual_weekend"] = df_check["is_weekend"].astype(bool)
weekend_mismatch = df_check[df_check["_expected_weekend"] != df_check["_actual_weekend"]]
print(f"is_weekend mismatches (DOW 5,6 vs is_weekend flag): {len(weekend_mismatch)}")
if len(weekend_mismatch) > 0:
    print("FAIL — Sample mismatches:")
    print(weekend_mismatch[["date", "day_of_week_number", "is_weekend"]].head(10))
else:
    print("PASS — is_weekend matches DOW 5,6 perfectly")

In [ ]:
# is_nerc_holiday: count per year (expect ~10-12)
df_check = df_dates.copy()
df_check["_year"] = pd.to_datetime(df_check["date"]).dt.year
holiday_counts = df_check[df_check["is_nerc_holiday"].astype(bool)].groupby("_year").size()
print("NERC holidays per year:")
print(holiday_counts.to_string())
print(f"\nMean per year: {holiday_counts.mean():.1f}")
print(f"Range: {holiday_counts.min()} to {holiday_counts.max()}")
outside_range = holiday_counts[(holiday_counts < 8) | (holiday_counts > 14)]
if len(outside_range) > 0:
    print(f"WARNING — Years with unusual holiday counts: {outside_range.to_dict()}")
else:
    print("PASS — All years have 8-14 NERC holidays")

In [ ]:
# summer_winter: verify seasonal boundaries make sense for PJM
sw_vals = df_dates["summer_winter"].unique()
print(f"summer_winter unique values: {sorted(sw_vals)}")

df_check = df_dates.copy()
df_check["_month"] = pd.to_datetime(df_check["date"]).dt.month
sw_by_month = df_check.groupby("_month")["summer_winter"].value_counts().unstack(fill_value=0)
print(f"\nSummer/Winter breakdown by month:")
print(sw_by_month.to_string())

## 4. NERC Holiday Analysis

In [ ]:
# List all NERC holidays
df_holidays = df_dates[df_dates["is_nerc_holiday"].astype(bool)].copy()
df_holidays["_date_ts"] = pd.to_datetime(df_holidays["date"])
df_holidays["_year"] = df_holidays["_date_ts"].dt.year
df_holidays["_month"] = df_holidays["_date_ts"].dt.month
df_holidays["_day"] = df_holidays["_date_ts"].dt.day
df_holidays["_dow_name"] = df_holidays["_date_ts"].dt.day_name()

print(f"Total NERC holidays: {len(df_holidays)}\n")

# Show holidays grouped by approximate type (month-based heuristic)
def classify_holiday(row):
    m, d = row["_month"], row["_day"]
    if m == 1 and d <= 3: return "New Year's Day"
    if m == 1 and d >= 15 and d <= 21: return "MLK Day"
    if m == 2 and d >= 15 and d <= 21: return "Presidents' Day"
    if m == 5 and d >= 25: return "Memorial Day"
    if m == 6 and d == 19: return "Juneteenth"
    if m == 7 and d <= 5: return "Independence Day"
    if m == 9 and d <= 7: return "Labor Day"
    if m == 10 and d >= 8 and d <= 14: return "Columbus Day"
    if m == 11 and d == 11: return "Veterans Day"
    if m == 11 and d >= 22 and d <= 28: return "Thanksgiving"
    if m == 12 and d >= 24: return "Christmas"
    return f"Unknown ({m}/{d})"

df_holidays["_holiday_name"] = df_holidays.apply(classify_holiday, axis=1)

# Show a sample from each holiday type
for name in df_holidays["_holiday_name"].unique():
    subset = df_holidays[df_holidays["_holiday_name"] == name]
    print(f"{name}: {len(subset)} occurrences")
    sample = subset[["date", "_dow_name"]].head(3)
    for _, r in sample.iterrows():
        print(f"    {r['date']} ({r['_dow_name']})")
    print()

In [ ]:
# Holiday counts per year bar chart
fig = px.bar(
    holiday_counts.reset_index(),
    x="_year",
    y=0,
    labels={"_year": "Year", 0: "NERC Holiday Count"},
    title="NERC Holidays per Year",
)
fig.add_hline(y=10, line_dash="dash", line_color="green", annotation_text="Expected ~10")
fig.update_layout(template="plotly_dark", height=400)
fig.show()

In [ ]:
# Check for holidays on weekends
weekend_holidays = df_holidays[df_holidays["day_of_week_number"].isin([5, 6])]
print(f"NERC holidays falling on weekends: {len(weekend_holidays)}")
if len(weekend_holidays) > 0:
    print(weekend_holidays[["date", "_dow_name", "_holiday_name"]].to_string(index=False))
else:
    print("None — all NERC holidays fall on weekdays")

## 5. Summer/Winter Transition

In [ ]:
# Plot summer_winter flag over time
df_sw = df_dates.copy()
df_sw["_date_ts"] = pd.to_datetime(df_sw["date"])
df_sw["_is_summer"] = (df_sw["summer_winter"].str.upper() == "SUMMER").astype(int)

fig = px.scatter(
    df_sw,
    x="_date_ts",
    y="_is_summer",
    color="summer_winter",
    labels={"_date_ts": "Date", "_is_summer": "Summer=1 / Winter=0", "summer_winter": "Season"},
    title="Summer/Winter Flag Over Time",
    opacity=0.4,
)
fig.update_layout(template="plotly_dark", height=400, yaxis=dict(tickvals=[0, 1], ticktext=["Winter", "Summer"]))
fig.show()

In [ ]:
# Identify summer/winter transition dates per year
df_sw_sorted = df_sw.sort_values("_date_ts").reset_index(drop=True)
df_sw_sorted["_prev_season"] = df_sw_sorted["summer_winter"].shift(1)
transitions = df_sw_sorted[df_sw_sorted["summer_winter"] != df_sw_sorted["_prev_season"]].dropna(subset=["_prev_season"])
transitions["_year"] = transitions["_date_ts"].dt.year

print("Summer/Winter transition dates:")
print(f"{'Date':<15} {'From':<10} {'To':<10}")
print("-" * 35)
for _, row in transitions.iterrows():
    print(f"{str(row['date']):<15} {row['_prev_season']:<10} {row['summer_winter']:<10}")

# Verify PJM summer is roughly Jun-Sep
summer_months = df_sw[df_sw["_is_summer"] == 1]["_date_ts"].dt.month.unique()
print(f"\nMonths flagged as SUMMER: {sorted(summer_months)}")
print(f"PJM summer typically Jun-Sep (months 6-9): {'Reasonable' if set(summer_months).issubset({5,6,7,8,9,10}) else 'Check boundaries'}")

## 6. Feature Sanity Check

In [ ]:
from src.like_day_forecast.features import calendar_features

df_feat = calendar_features.build(df_dates)

print(f"Feature shape: {df_feat.shape}")
print(f"Feature columns ({len(df_feat.columns)}):")
print(f"  {list(df_feat.columns)}")
n_features = len([c for c in df_feat.columns if c != "date"])
print(f"\nNumber of features (excl. date): {n_features}")
print(f"Expected 17: {'PASS' if n_features == 17 else 'FAIL'}")
df_feat.head()

In [ ]:
# Verify DOW group assignments match config
df_verify = df_dates[["date", "day_of_week_number"]].merge(df_feat[["date", "dow_group"]], on="date")

# Build expected group map from configs
expected_map = {}
for group_idx, (group_name, days) in enumerate(configs.DOW_GROUPS.items()):
    for day in days:
        expected_map[day] = group_idx

df_verify["_expected_group"] = df_verify["day_of_week_number"].map(expected_map)
group_mismatch = df_verify[df_verify["dow_group"] != df_verify["_expected_group"]]
print(f"DOW group mismatches: {len(group_mismatch)}")
if len(group_mismatch) > 0:
    print("FAIL — Sample mismatches:")
    print(group_mismatch.head())
else:
    print("PASS — All DOW group assignments match config")

# Show the group mapping
print(f"\nDOW group mapping (from configs.DOW_GROUPS):")
for group_idx, (group_name, days) in enumerate(configs.DOW_GROUPS.items()):
    day_labels = [["Mon","Tue","Wed","Thu","Fri","Sat","Sun"][d] for d in days]
    print(f"  Group {group_idx} ({group_name}): DOW {days} = {day_labels}")

In [ ]:
# Verify cyclical encodings: sin^2 + cos^2 ≈ 1 for each pair
cyclical_pairs = [
    ("dow_sin", "dow_cos"),
    ("month_sin", "month_cos"),
    ("day_of_year_sin", "day_of_year_cos"),
]

print("Cyclical encoding sin^2 + cos^2 check:")
for sin_col, cos_col in cyclical_pairs:
    magnitude = df_feat[sin_col]**2 + df_feat[cos_col]**2
    max_err = (magnitude - 1.0).abs().max()
    mean_err = (magnitude - 1.0).abs().mean()
    print(f"  {sin_col}/{cos_col}: max_err={max_err:.2e}, mean_err={mean_err:.2e} — {'PASS' if max_err < 1e-10 else 'FAIL'}")

# Verify one-hot DOW sums to 1 per row
dow_cols = [f"dow_{d}" for d in range(7)]
onehot_sums = df_feat[dow_cols].sum(axis=1)
all_one = (onehot_sums == 1).all()
print(f"\nOne-hot DOW sum == 1 for all rows: {'PASS' if all_one else 'FAIL'}")
if not all_one:
    bad_rows = df_feat[onehot_sums != 1]
    print(f"  Bad rows: {len(bad_rows)}")
    print(bad_rows[["date"] + dow_cols].head())

## 7. Cyclical Encoding Visualization

In [ ]:
# DOW cyclical encoding — should form 7 distinct clusters on a circle
df_plot = df_feat.copy()
df_plot["_dow"] = df_dates.sort_values("date").reset_index(drop=True)["day_of_week_number"]
day_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
df_plot["_day_name"] = df_plot["_dow"].map(lambda x: day_names[x])

fig = px.scatter(
    df_plot,
    x="dow_cos",
    y="dow_sin",
    color="_day_name",
    title="Day-of-Week Cyclical Encoding (should form circle with 7 clusters)",
    labels={"dow_cos": "cos(2*pi*DOW/7)", "dow_sin": "sin(2*pi*DOW/7)", "_day_name": "Day"},
    category_orders={"_day_name": day_names},
    opacity=0.5,
)
fig.update_layout(template="plotly_dark", height=500, width=600, yaxis_scaleanchor="x")
fig.show()

In [ ]:
# Day-of-year cyclical encoding — should form a smooth circle colored by month
df_plot2 = df_feat.copy()
df_plot2["_month"] = pd.to_datetime(df_plot2["date"]).dt.month
month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
df_plot2["_month_name"] = df_plot2["_month"].map(lambda x: month_names[x - 1])

fig = px.scatter(
    df_plot2,
    x="day_of_year_cos",
    y="day_of_year_sin",
    color="_month_name",
    title="Day-of-Year Cyclical Encoding (should form smooth circle)",
    labels={
        "day_of_year_cos": "cos(2*pi*DOY/365.25)",
        "day_of_year_sin": "sin(2*pi*DOY/365.25)",
        "_month_name": "Month",
    },
    category_orders={"_month_name": month_names},
    opacity=0.4,
)
fig.update_layout(template="plotly_dark", height=500, width=600, yaxis_scaleanchor="x")
fig.show()

## 8. DOW Group Distribution

In [ ]:
# DOW group distribution bar chart
group_names = list(configs.DOW_GROUPS.keys())
group_counts = df_feat["dow_group"].value_counts().sort_index()

df_bar = pd.DataFrame({
    "group_idx": group_counts.index,
    "group_name": [group_names[i] for i in group_counts.index],
    "count": group_counts.values,
})
df_bar["pct"] = (df_bar["count"] / df_bar["count"].sum() * 100).round(1)
df_bar["label"] = df_bar.apply(lambda r: f"{r['group_name']} (DOW {configs.DOW_GROUPS[r['group_name']]})", axis=1)

fig = px.bar(
    df_bar,
    x="label",
    y="count",
    text="pct",
    title="DOW Group Distribution Across Pool",
    labels={"label": "DOW Group", "count": "Number of Days", "pct": "% of Total"},
)
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_layout(template="plotly_dark", height=450)
fig.show()

print("\nGroup balance summary:")
for _, row in df_bar.iterrows():
    print(f"  {row['label']}: {row['count']:,} days ({row['pct']}%)")

# Check if groups are reasonably balanced for analog matching
max_pct = df_bar["pct"].max()
min_pct = df_bar["pct"].min()
print(f"\nMax/Min group % ratio: {max_pct/min_pct:.1f}x")
print(f"Note: weekday_early (3 days) expected ~3/7=43%, saturday/sunday (1 day each) ~14%")

## 9. Recent Spot Check

In [ ]:
# Last 14 days — raw dates data with all columns
df_recent_raw = df_dates.sort_values("date").tail(14).copy()
df_recent_raw["_dow_name"] = pd.to_datetime(df_recent_raw["date"]).dt.day_name()
print("Last 14 days — raw calendar data:")
df_recent_raw[["date", "_dow_name", "day_of_week_number", "is_weekend", "is_nerc_holiday", "summer_winter"]]

In [ ]:
# Last 14 days — built features
df_feat_recent = df_feat.sort_values("date").tail(14)
print("Last 14 days — calendar features:")
df_feat_recent

## 10. Validation Summary

In [ ]:
# ---- Validation Summary ----
date_series = pd.to_datetime(df_dates["date"])
full_range = pd.date_range(start=date_series.min(), end=date_series.max(), freq="D")
missing_dates = full_range.difference(date_series)

# Cyclical checks
cyclical_pairs = [
    ("dow_sin", "dow_cos"),
    ("month_sin", "month_cos"),
    ("day_of_year_sin", "day_of_year_cos"),
]
cyclical_ok = True
for sin_col, cos_col in cyclical_pairs:
    mag = df_feat[sin_col]**2 + df_feat[cos_col]**2
    if (mag - 1.0).abs().max() > 1e-10:
        cyclical_ok = False

# One-hot check
dow_cols = [f"dow_{d}" for d in range(7)]
onehot_ok = (df_feat[dow_cols].sum(axis=1) == 1).all()

# Weekend check
df_wk = df_dates.copy()
wk_match = (df_wk["is_weekend"].astype(bool) == df_wk["day_of_week_number"].isin([5, 6])).all()

# NERC holiday count check
df_hc = df_dates.copy()
df_hc["_year"] = pd.to_datetime(df_hc["date"]).dt.year
hol_counts = df_hc[df_hc["is_nerc_holiday"].astype(bool)].groupby("_year").size()
nerc_ok = ((hol_counts >= 8) & (hol_counts <= 14)).all()

checks = {
    "Date range covers 2014":           date_series.min() <= pd.Timestamp("2014-12-31"),
    "One row per date":                  len(df_dates) == len(full_range),
    "No missing calendar dates":         len(missing_dates) == 0,
    "day_of_week_number in [0,6]":       set(df_dates["day_of_week_number"].unique()) == set(range(7)),
    "is_weekend matches DOW 5,6":        wk_match,
    "NERC holidays 8-14 per year":       nerc_ok,
    "sin^2+cos^2 ~ 1 for all cyclical": cyclical_ok,
    "One-hot DOW sums to 1":            onehot_ok,
}

print("=" * 55)
print("  VALIDATION SUMMARY")
print("=" * 55)
all_pass = True
for check_name, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    if not passed:
        all_pass = False
    print(f"  [{status}]  {check_name}")
print("=" * 55)
print(f"  Overall: {'ALL CHECKS PASSED' if all_pass else 'SOME CHECKS FAILED'}")
print("=" * 55)